## 获取待提取特征的文件

提供两种批量处理的模式：
1. 目录模式，提取指定目录下的所有jpg文件的特征。
2. 文件模式，待提取的数据存储在文件中，每行一个样本。

当然也可以在最后自己指定手动提取指定若干文件。

In [1]:
from functools import partial
import os
import pandas as pd
from glob import glob 
from onekey_algo import get_param_in_cwd
from onekey_algo.custom.components.comp1 import compress_df_feature
from onekey_algo.custom.components.comp2 import extract, print_feature_hook, reg_hook_on_module, \
    init_from_model, init_from_onekey, feature_layer_mapping

samples = glob(os.path.join(r'Y:\20251014-fenghexin_fuxin\L', '*'))
print(len(samples))
model_root = r'Y:\20251014-fenghexin_fuxin\Lmodels\CV-0\resnet50'
model, transformer, device = init_from_onekey(os.path.join(model_root, 'viz'), 
                                             force_pth=r'Y:/20251014-fenghexin_fuxin/Lmodels/CV-0/resnet50/train/training-params-50.pth')
for n, m in model.named_modules():
    print('Feature name:', n, "|| Module:", m)

os.makedirs('features', exist_ok=True)
feature_name = feature_layer_mapping[os.path.basename(model_root) + '_2D']
with open(f'features/DL_feature.csv', 'w', encoding='utf-8-sig') as outfile:
    hook = partial(print_feature_hook, fp=outfile)
    find_num = reg_hook_on_module(feature_name, model, hook)
    results = extract(samples, model, transformer, device, fp=outfile)

features = pd.read_csv(f'features/DL_feature.csv', header=None)
features.columns=['ID'] + list(f"DL_{c}" for c in features.columns[1:])
features.to_csv(f'features/DL_feature.csv', index=False, encoding='utf-8-sig')

cm_features = compress_df_feature(features=features, dim=16, prefix='DL_', not_compress='ID')
cm_features.to_csv(f'features/DL_compress_features.csv', header=True, index=False, encoding='utf-8-sig')

[2026-02-16 09:48:28 - comp2.py: 214]	INFO	Using force param files: Y:/20251014-fenghexin_fuxin/Lmodels/CV-0/resnet50/train/training-params-50.pth
[2026-02-16 09:48:28 - comp2.py: 235]	INFO	模型参数：{'pretrained': False, 'model_name': 'resnet50', 'num_classes': 1, 'in_channels': 3}


291
Feature name:  || Module: ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, k